# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samhere3116-maker/ML-Pipeline/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane predicts a yes/no label (is_declining) that I already defined and validated in Week 3, so per the toolkit table this is "yes/no with an observed label" → start with Logistic Regression (readable), then compare against Random Forest (stronger). Since the real output is a ranked queue, I evaluate both using Precision@50, the same metric as my Week-4 baseline.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [9]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY90 = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
MONTH_START, MONTH_END = '2026-03-01', '2026-03-31'

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

CUTOFF = "2026-03-01"          # last30 vs prev30 split point
QUERY_START = "2026-02-01"     # pulling Feb + March so prev30 has a real ~29-day span
MONTH_END = "2026-03-31"

# --- Build feature frame from the warehouse ---
feat_frame_full = con.sql(f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date > DATE '{CUTOFF}' THEN gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date <= DATE '{CUTOFF}' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN report_date <= DATE '{CUTOFF}' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN report_date <= DATE '{CUTOFF}' THEN gsc_avg_position END) AS pos_prev30
        FROM {FACT}
        WHERE report_date BETWEEN '{QUERY_START}' AND '{MONTH_END}'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 50
    ) SELECT * FROM windowed
""").df()

# --- Add query signal + content age ---
qsignals = con.sql(f"""
    SELECT content_hash_id, ANY_VALUE(content_visible_query_count) AS visible_queries
    FROM {QUERY90} GROUP BY content_hash_id
""").df()

content_age = con.sql(f"""
    SELECT content_hash_id, DATE_DIFF('day', content_created_date, DATE '{MONTH_END}') AS content_age_days
    FROM {DIM_CONTENT}
""").df()

feat_frame_full = feat_frame_full.merge(qsignals, on='content_hash_id', how='left').merge(content_age, on='content_hash_id', how='left')

# --- Label ---
feat_frame_full['is_declining'] = (feat_frame_full['imp_last30'] < 0.8 * feat_frame_full['imp_prev30']).astype(int)

# --- Merge in Week 4 baseline score ---
baseline_url = "https://raw.githubusercontent.com/samhere3116-maker/ML-Pipeline/main/outputs/baseline_action_score.csv"
baseline = pd.read_csv(baseline_url)[['client_hash_id','content_hash_id','baseline_action_score']]
feat_frame_full = feat_frame_full.merge(baseline, on=['client_hash_id','content_hash_id'], how='left')
feat_frame_full['baseline_action_score'] = feat_frame_full['baseline_action_score'].fillna(0)

# --- Build modeling set ---
cols = ['imp_prev30','clk_prev30','pos_prev30','visible_queries','content_age_days']
model_df = feat_frame_full.dropna(subset=cols).reset_index(drop=True)
print(f"{len(model_df):,} rows in modeling set | base rate (declining): {model_df['is_declining'].mean():.3%}")

# --- Grouped split by client ---
X, y, groups = model_df[cols], model_df['is_declining'], model_df['client_hash_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

overlap = set(model_df.iloc[train_idx]['client_hash_id']) & set(model_df.iloc[test_idx]['client_hash_id'])
print(f"Train rows: {len(train_idx):,} | Test rows: {len(test_idx):,} | Clients overlapping both: {len(overlap)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

73,112 rows in modeling set | base rate (declining): 18.213%
Train rows: 58,049 | Test rows: 15,063 | Clients overlapping both: 0


I split by client_hash_id, not by row, using a 75/25 group split — 58,049 train rows and 15,063 test rows across 73,112 total content items, with zero clients appearing in both sets. (Note: my first attempt at building the label only used March's 31 days split in half, which gave the "previous" window just 1 day of data — I caught this because it made almost every page look like it was improving, with a base rate near 0%. Pulling Feb+March fixed it to a realistic 18.2% decline rate.) I split by client rather than by row because pages from the same client share template, niche, and overall site health — a random split would let client-specific patterns leak from train into test.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
baseline_scores_test = model_df.iloc[test_idx]['baseline_action_score'].values

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return np.array(y_true)[order].mean()

logreg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced', random_state=42).fit(X_train, y_train)

comparison = pd.DataFrame({
    'method': ['base_rate', 'Week-4 baseline rule', 'Logistic Regression', 'Random Forest'],
    'precision_at_50': [
        y_test.mean(),
        precision_at_k(y_test, baseline_scores_test),
        precision_at_k(y_test, logreg.predict_proba(X_test)[:,1]),
        precision_at_k(y_test, rf.predict_proba(X_test)[:,1]),
    ]
})
comparison

,method,precision_at_50
0,base_rate,0.133174
1,Week-4 baseline rule,0.000000
2,Logistic Regression,0.380000
3,Random Forest,0.320000


Precision@50 — base rate: 0.133, Week-4 baseline: 0.000, Logistic Regression: 0.380, Random Forest: 0.320. The baseline rule scoring zero isn't a fair "loss" — it was built to catch CTR-vs-position problems, not declining impressions, so the two targets barely overlap. Both trained models beat the random base rate substantially, with Logistic Regression outperforming Random Forest despite being the simpler model — with only 5 features, the extra flexibility of a Random Forest doesn't appear to earn its keep here. This is the actual lesson of the week: a more complex model isn't automatically a better one.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importances = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

test_df = model_df.iloc[test_idx].reset_index(drop=True).copy()
test_df['logreg_score'] = logreg.predict_proba(X_test)[:,1]
test_df['true_label'] = y_test

false_positives = test_df[(test_df['logreg_score'] > 0.5) & (test_df['true_label'] == 0)].sort_values('logreg_score', ascending=False).head(3)
print("\nTop false positives (model confident, but page wasn't actually declining):")
false_positives[cols + ['logreg_score','true_label']]

Random Forest feature importances:
visible_queries     0.337445
content_age_days    0.260472
imp_prev30          0.255593
pos_prev30          0.109038
clk_prev30          0.037452
dtype: float64

Top false positives (model confident, but page wasn't actually declining):


,imp_prev30,clk_prev30,pos_prev30,visible_queries,content_age_days,logreg_score,true_label
14231,90512.0,13.0,5.195844,134.0,69,1.000000,0
384,81900.0,76.0,20.114995,204.0,263,0.999981,0
418,49443.0,188.0,5.362374,8.0,263,0.999823,0


The Random Forest leaned most heavily on visible_queries (0.337) and content_age_days (0.260), with imp_prev30 close behind (0.256); pos_prev30 (0.109) and especially clk_prev30 (0.037) contributed comparatively little. I'd expected click behavior to matter more, so this was worth noting rather than assuming beforehand.

Looking at the false positives, all three share unusually high imp_prev30 (49K–90K), well above a typical page in this dataset. This suggests the model may be more prone to misfiring on very high-traffic pages — a 10% fluctuation on 90,000 impressions is a large number in absolute terms even if it's within normal month-to-month variance for a page that size. A stronger version of this label might use a percentage-based threshold scaled to a page's own traffic history, rather than one fixed 20% cutoff applied uniformly to pages of very different sizes.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.